# Aula 2, História da IA

Notebook executável que acompanha a aula [02-historia-da-ia.md](../../lessons/modulo-01-introducao-ia/02-historia-da-ia.md).

## O que vamos fazer aqui

A história da IA tem um momento decisivo: em 1969, Minsky e Papert mostraram que o
perceptron, um dos primeiros modelos que aprendem, não conseguia resolver um
problema tão simples quanto o XOR. Esse resultado ajudou a provocar o primeiro
inverno da IA.

Neste notebook vamos reviver esse episódio com código. Primeiro implementamos o
perceptron e o treinamos na função AND, que ele resolve. Depois tentamos o XOR, que
ele não consegue aprender. Por fim, no projeto, usamos uma rede de múltiplas camadas
para vencer a mesma limitação, repetindo o caminho que a área seguiu na prática.

## O perceptron

O perceptron calcula uma soma ponderada das entradas, aplica uma função degrau e
ajusta os pesos sempre que erra. As funções abaixo implementam exatamente isso, com
a regra de atualização clássica. Só precisamos do numpy.

In [ ]:
import numpy as np


def degrau(z):
    """Função de ativação do perceptron: 1 se z >= 0, senão 0."""
    return np.where(z >= 0, 1, 0)


def treinar_perceptron(X, y, epocas=20, taxa=0.1):
    """Treina um perceptron simples com a regra clássica de atualização."""
    n_features = X.shape[1]
    w = np.zeros(n_features)  # pesos começam em zero
    b = 0.0                   # viés começa em zero
    for _ in range(epocas):
        for xi, yi in zip(X, y):
            z = np.dot(w, xi) + b
            y_previsto = degrau(z)
            erro = yi - y_previsto
            # Só ajusta quando erra. Se acertou, erro é zero e nada muda.
            w = w + taxa * erro * xi
            b = b + taxa * erro
    return w, b

## Tarefa 1, AND (linearmente separável)

A função AND vale 1 só quando as duas entradas são 1. Esse caso é linearmente
separável, ou seja, existe uma reta que separa o único exemplo positivo dos demais,
e o perceptron consegue encontrá-la.

In [ ]:
# As quatro combinações possíveis de duas entradas binárias.
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])

y_and = np.array([0, 0, 0, 1])
w, b = treinar_perceptron(X, y_and)
print('AND  previsto:', degrau(X @ w + b), '| esperado:', y_and)

O perceptron acerta o AND em cheio. Como existe uma reta que separa as
classes, a regra de atualização converge para uma solução correta.

## Tarefa 2, XOR (não é linearmente separável)

O XOR vale 1 quando as entradas são diferentes. Aqui não existe uma única reta que
separe os casos positivos dos negativos, e é isso que vai derrubar o perceptron.

In [ ]:
y_xor = np.array([0, 1, 1, 0])
w, b = treinar_perceptron(X, y_xor)
print('XOR  previsto:', degrau(X @ w + b), '| esperado:', y_xor)

O perceptron erra o XOR, e não melhora por mais épocas que você dê. Isso não
é um bug do código, é a limitação matemática real apontada por Minsky e Papert em
1969. Um modelo que só traça uma reta não dá conta de uma fronteira em forma de X.

## Projeto da aula, superando o limite com uma rede de múltiplas camadas

A solução histórica foi empilhar camadas e treiná-las com backpropagation. A célula
abaixo resolve o mesmo XOR com uma rede de uma camada escondida, usando o
scikit-learn. Compare o resultado com o do perceptron acima.

In [ ]:
from sklearn.neural_network import MLPClassifier

rede = MLPClassifier(hidden_layer_sizes=(4,), activation='tanh',
                     max_iter=5000, random_state=0)
rede.fit(X, y_xor)
print('XOR pela rede:', rede.predict(X), '| esperado:', y_xor)

A rede acerta as quatro combinações do XOR. A camada escondida permite
combinar duas fronteiras, criando a separação em X que uma única reta não fazia. Para
o projeto, escreva um parágrafo explicando, com suas palavras, o que a camada
escondida acrescenta, ligando com a virada que a backpropagation trouxe em 1986.